In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from src.features.engineering import load_processed # Ajusta la ruta si es necesario

# 1. Cargar los datos combinados (simulando tu load_multiple)
tickers = ["AAPL", "MSFT", "GOOGL", "JPM", "SPY"]
dfs = []
for ticker in tickers:
    df = load_processed(ticker)
    df["ticker"] = ticker
    dfs.append(df)
df_combined = pd.concat(dfs).sort_index()

# 2. Separar Features y Target
cols_to_exclude = ["Open", "High", "Low", "Close", "Volume", "target", "log_return_10d", "ticker"]
features = [c for c in df_combined.columns if c not in cols_to_exclude]

# IMPORTANTE: Usamos TODO el dataset excepto el final absoluto (Test real)
# Dejaremos el último año fuera del GridSearch para no contaminar el test final.
train_mask = df_combined.index <= "2022-12-31" 
X_train_grid = df_combined.loc[train_mask, features]
y_train_grid = df_combined.loc[train_mask, "target"]

# 3. Configurar la validación cruzada temporal (5 cortes hacia adelante)
tscv = TimeSeriesSplit(n_splits=5)

print(f"Filas para GridSearch: {len(X_train_grid)}")

Filas para GridSearch: 8810


In [2]:
print("Iniciando búsqueda para Random Forest...")

# Instancia base (fijamos lo que no va a cambiar)
rf_base = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

# La red de posibilidades a probar
param_grid_rf = {
    'n_estimators': [200, 500],          # Cuántos árboles
    'max_depth': [2, 3, 5],              # Qué tan profundos (5 ya es mucho para finanzas)
    'min_samples_leaf': [30, 50, 100],   # Hojas gordas = menos ruido
    'max_features': ['sqrt', 0.5]        # Cuántas variables ve cada árbol
}

# Configurar el buscador
grid_rf = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid_rf,
    cv=tscv,                 # La validación temporal
    scoring='roc_auc',       # Nuestra métrica de oro
    verbose=2,               # Para que imprima el progreso en pantalla
    n_jobs=-1                # Usar todos los núcleos de tu procesador
)

# ¡A correr!
grid_rf.fit(X_train_grid, y_train_grid)

print("\n" + "="*50)
print("🏆 MEJOR RANDOM FOREST 🏆")
print(f"Mejor ROC-AUC (Cross-Val): {grid_rf.best_score_:.4f}")
print("Mejores parámetros:")
for k, v in grid_rf.best_params_.items():
    print(f"  - {k}: {v}")
print("="*50)

Iniciando búsqueda para Random Forest...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

🏆 MEJOR RANDOM FOREST 🏆
Mejor ROC-AUC (Cross-Val): 0.5309
Mejores parámetros:
  - max_depth: 2
  - max_features: 0.5
  - min_samples_leaf: 30
  - n_estimators: 200


In [3]:
print("Iniciando búsqueda para XGBoost...")

# Calcular el desbalance global
scale = (y_train_grid == 0).sum() / (y_train_grid == 1).sum()

# Instancia base
xgb_base = xgb.XGBClassifier(
    scale_pos_weight=scale,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    verbosity=0
)

# La red de posibilidades (Enfocada en frenar el sobreajuste)
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [1, 2, 3],               # Arboles casi ciegos para no memorizar
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.5, 0.8],              # Inyectar caos (usar 50% u 80% de datos)
    'colsample_bytree': [0.5, 0.8],
    'reg_alpha': [0.1, 0.5, 1.0],         # Poda matemática
    'min_child_weight': [10, 30]
}

grid_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_xgb,
    cv=tscv,
    scoring='roc_auc',
    verbose=1,
    n_jobs=-1
)

# ¡A correr!
grid_xgb.fit(X_train_grid, y_train_grid)

print("\n" + "="*50)
print("🏆 MEJOR XGBOOST 🏆")
print(f"Mejor ROC-AUC (Cross-Val): {grid_xgb.best_score_:.4f}")
print("Mejores parámetros:")
for k, v in grid_xgb.best_params_.items():
    print(f"  - {k}: {v}")
print("="*50)

Iniciando búsqueda para XGBoost...
Fitting 5 folds for each of 648 candidates, totalling 3240 fits

🏆 MEJOR XGBOOST 🏆
Mejor ROC-AUC (Cross-Val): 0.5384
Mejores parámetros:
  - colsample_bytree: 0.5
  - learning_rate: 0.01
  - max_depth: 1
  - min_child_weight: 10
  - n_estimators: 100
  - reg_alpha: 1.0
  - subsample: 0.5
